# clone_voice

가족의 목소리를 한 번 등록해 두면, 이후에는 문장만 입력해서 그 목소리로 음성을 만든다.
Qwen3-TTS Base 모델을 쓰고, 대본은 Whisper가 자동으로 받아쓴다.

## 실행 순서

1. **런타임 → 런타임 유형 변경 → GPU** 로 설정
2. 아래 셀 4개를 위에서부터 순서대로 실행
3. 마지막 셀이 출력하는 주소를 새 탭에서 열고 로그인

처음 실행하면 모델을 내려받느라 몇 분 걸린다. 두 번째부터는 빠르다.

## 로그인이 걸려 있다

UI는 `*.gradio.live` 주소로 열리지만 아이디·비밀번호 없이는 아무것도 볼 수 없다.
로그인 정보는 Colab 보안 비밀에서 읽거나 세션마다 새로 만들며, 노트북 파일에는 저장되지 않는다.

원리와 파일 규칙, 문제 해결은 저장소의 `README.md`에 정리돼 있다.

## 1. Google Drive 연결

등록한 목소리와 생성 결과를 Drive에 보관한다. 세션이 끊겨도 남는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설치

`qwen-tts`가 `transformers`를 함께 설치하므로 Whisper용 추가 패키지는 없다.
런타임 재시작을 요구하면 재시작하고 **1번부터** 다시 실행한다.

In [ ]:
!pip install qwen-tts soundfile gradio
!apt-get -y install ffmpeg

## 3. 설정과 모델 로드

경로를 바꾸려면 `VOICE_DIR` / `OUTPUT_DIR`만 수정한다.
TTS 모델은 여기서 로드하고, Whisper는 대본이 없을 때만 로드한다.

In [ ]:
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

# ── 설정 ────────────────────────────────────────────────
VOICE_DIR = Path("/content/drive/MyDrive/2026/clone_voice/recored_voice")   # 등록한 목소리
OUTPUT_DIR = Path("/content/drive/MyDrive/2026/clone_voice/output")          # 생성 결과
LANGUAGE = "Korean"

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
STT_MODEL_ID = "openai/whisper-large-v3-turbo"
STT_LANGUAGE = "ko"

AUDIO_EXTS = [".m4a", ".mp3", ".wav", ".webm", ".ogg", ".flac"]
VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".avi"]
MEDIA_EXTS = AUDIO_EXTS + VIDEO_EXTS
CACHE_SUFFIX = "_16k"      # 변환 캐시 파일 표시
MIN_DURATION = 3.0         # 참조 음성 최소 권장 길이(초)
# ───────────────────────────────────────────────────────

VOICE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32


# ── 파일 ────────────────────────────────────────────────

def find_media(name):
    """이름에 해당하는 참조 파일(음성 또는 영상)을 찾는다."""
    for ext in MEDIA_EXTS:
        p = VOICE_DIR / f"{name}{ext}"
        if p.exists():
            return p
    return None


def list_voices():
    """등록된 목소리 이름 목록. 변환 캐시는 제외한다."""
    names = []
    for p in sorted(VOICE_DIR.iterdir()):
        if p.suffix.lower() in MEDIA_EXTS and not p.stem.endswith(CACHE_SUFFIX):
            if p.stem not in names:
                names.append(p.stem)
    return names


def read_text(path):
    """한글 txt는 UTF-8과 CP949가 섞여 있어 순서대로 시도한다."""
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return path.read_text(encoding=enc).strip()
        except UnicodeDecodeError:
            continue
    raise ValueError(f"'{path.name}' 인코딩을 인식하지 못했습니다.")


def duration_of(wav_path):
    data, sr = sf.read(wav_path)
    return len(data) / sr


# ── 변환 ────────────────────────────────────────────────

def has_audio(src):
    """오디오 트랙이 있는지 확인한다 (무음 영상 대비)."""
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a",
         "-show_entries", "stream=codec_type", "-of", "csv=p=0", str(src)],
        capture_output=True, text=True,
    )
    return "audio" in r.stdout


def to_wav(src):
    """16kHz 모노 wav로 변환한다. 캐시가 원본보다 최신이면 건너뛴다."""
    dst = src.with_name(f"{src.stem}{CACHE_SUFFIX}.wav")
    if dst.exists() and dst.stat().st_mtime >= src.stat().st_mtime:
        return dst
    if not has_audio(src):
        raise ValueError(f"'{src.name}'에 오디오 트랙이 없습니다.")
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-vn", "-ac", "1", "-ar", "16000", str(dst)],
        check=True, capture_output=True,
    )
    return dst


# ── STT ─────────────────────────────────────────────────

_stt_pipe = None


def get_stt():
    """Whisper를 처음 필요할 때만 로드한다."""
    global _stt_pipe
    if _stt_pipe is None:
        from transformers import pipeline
        print(f"STT 모델 로드 중: {STT_MODEL_ID}")
        _stt_pipe = pipeline(
            "automatic-speech-recognition",
            model=STT_MODEL_ID,
            device=0 if torch.cuda.is_available() else -1,
            dtype=DTYPE,
        )
    return _stt_pipe


def transcribe(wav_path):
    """참조 음성을 받아써서 문자열로 돌려준다."""
    result = get_stt()(
        str(wav_path),
        chunk_length_s=30,
        generate_kwargs={"language": STT_LANGUAGE, "task": "transcribe"},
    )
    return result["text"].strip()


# ── 참조 로드 ────────────────────────────────────────────

def load_reference(name):
    """이름으로 (원본, 16k wav, 대본)을 가져온다. 대본이 없으면 STT로 만들어 저장한다."""
    media = find_media(name)
    if media is None:
        raise FileNotFoundError(f"'{name}' 참조 파일이 없습니다.")
    wav = to_wav(media)
    txt = VOICE_DIR / f"{name}.txt"
    if txt.exists():
        ref_text = read_text(txt)
    else:
        ref_text = transcribe(wav)
        txt.write_text(ref_text + "\n", encoding="utf-8")
    return media, wav, ref_text


# ── 모델 ────────────────────────────────────────────────

print("DEVICE:", DEVICE)
model = Qwen3TTSModel.from_pretrained(MODEL_ID, device_map=DEVICE, dtype=DTYPE)
print("TTS 모델 로드 완료:", MODEL_ID)
print("등록된 목소리:", list_voices() or "없음")

## 4. UI 실행

이 셀을 실행하면 `https://xxxx.gradio.live` 주소가 출력된다. **새 탭에서 열고** 로그인하면 UI가 나온다.
셀 안에서 열면 브라우저가 쿠키를 막아 로그인이 반복되므로 반드시 새 탭을 쓴다.

**로그인 정보** — Colab 왼쪽 사이드바의 🔑 **보안 비밀**에 `UI_USER`와 `UI_PASS`를 등록해 두면
매번 그 값으로 로그인한다. 등록하지 않으면 세션마다 임시 비밀번호를 만들어 셀에 출력한다.
비밀번호는 노트북 파일에 저장되지 않으므로 GitHub에 올라가지 않는다.

**목소리 등록** — 이름을 적고 마이크로 녹음하거나 파일을 올린다. 대본을 비워 두면
Whisper가 받아쓴다. 결과가 화면에 표시되니 틀린 곳은 고쳐서 "대본만 다시 저장"을 누른다.

**음성 생성** — 목록에서 고르고 문장을 입력한다. 결과는 `output` 폴더에 저장된다.

`*.gradio.live` 주소는 런타임이 살아 있는 동안만 유효하고, 로그인 없이는 아무것도 볼 수 없다.

In [ ]:
import gradio as gr


def ui_refresh():
    names = list_voices()
    return gr.update(choices=names, value=names[0] if names else None)


def ui_info(name):
    """선택한 목소리의 참조 음성과 대본을 보여준다. STT는 돌리지 않는다."""
    if not name:
        return None, ""
    media = find_media(name)
    txt = VOICE_DIR / f"{name}.txt"
    text = read_text(txt) if txt.exists() else "(대본 없음 — 생성할 때 자동으로 만들어집니다)"
    return (str(media) if media else None), text


def ui_register(name, audio_path, ref_text, overwrite):
    """마이크 녹음 또는 업로드 파일을 목소리로 등록한다."""
    name = (name or "").strip()
    if not name:
        return "이름을 입력하세요.", "", gr.update()
    if any(c in name for c in "/\\"):
        return "이름에 / 또는 \\ 를 쓸 수 없습니다.", "", gr.update()
    if not audio_path:
        return "마이크로 녹음하거나 파일을 올려주세요.", "", gr.update()

    existing = find_media(name)
    if existing and not overwrite:
        return f"'{name}'이 이미 있습니다. 덮어쓰려면 아래 체크박스를 켜세요.", "", gr.update()

    dst = None
    try:
        src = Path(audio_path)
        ext = src.suffix.lower() if src.suffix.lower() in MEDIA_EXTS else ".wav"
        dst = VOICE_DIR / f"{name}{ext}"
        shutil.copy(src, dst)
        wav = to_wav(dst)
        dur = duration_of(wav)

        text = (ref_text or "").strip()
        if text:
            src_label = "직접 입력"
        else:
            text = transcribe(wav)
            src_label = "STT 자동 생성"
        (VOICE_DIR / f"{name}.txt").write_text(text + "\n", encoding="utf-8")
    except Exception as e:
        # 새로 등록하다 실패한 경우에만 복사본을 지운다 (기존 등록은 건드리지 않음)
        if dst is not None and existing is None and dst.exists():
            dst.unlink()
        return f"등록 실패: {e}", "", gr.update()

    msg = f"등록 완료 — {name} ({dur:.1f}초), 대본 {src_label}"
    if dur < MIN_DURATION:
        msg += f"\n주의: {MIN_DURATION}초 미만이라 클로닝 품질이 떨어질 수 있습니다."
    names = list_voices()
    return msg, text, gr.update(choices=names, value=name)


def ui_save_text(name, text):
    """대본만 수정해서 다시 저장한다."""
    name = (name or "").strip()
    text = (text or "").strip()
    if not name:
        return "이름을 입력하세요."
    if not text:
        return "대본이 비어 있습니다."
    if not find_media(name):
        return f"'{name}'으로 등록된 목소리가 없습니다."
    (VOICE_DIR / f"{name}.txt").write_text(text + "\n", encoding="utf-8")
    return f"대본 저장 완료 — {name}.txt"


def ui_generate(name, target_text):
    """선택한 목소리로 문장을 합성한다."""
    if not name:
        return None, "목소리를 선택하세요."
    target_text = (target_text or "").strip()
    if not target_text:
        return None, "생성할 문장을 입력하세요."

    try:
        _, wav, ref_text = load_reference(name)
        wavs, out_sr = model.generate_voice_clone(
            text=target_text,
            language=LANGUAGE,
            ref_audio=str(wav),
            ref_text=ref_text,
        )
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = OUTPUT_DIR / f"{name}_{stamp}.wav"
        sf.write(out_path, wavs[0], out_sr)
    except Exception as e:
        return None, f"생성 실패: {e}"

    return str(out_path), f"완료 — {out_path.name}"


_names = list_voices()

with gr.Blocks(title="clone_voice") as demo:
    gr.Markdown("# clone_voice\n목소리를 한 번 등록해 두면 문장만 바꿔 가며 생성할 수 있습니다.")

    with gr.Tabs():
        with gr.Tab("음성 생성"):
            with gr.Row():
                gen_name = gr.Dropdown(
                    choices=_names,
                    value=_names[0] if _names else None,
                    label="목소리",
                    scale=4,
                )
                gen_refresh = gr.Button("목록 새로고침", scale=1)

            gen_ref_audio = gr.Audio(label="참조 음성", interactive=False)
            gen_ref_text = gr.Textbox(label="참조 대본", interactive=False, lines=2)

            gen_text = gr.Textbox(
                label="생성할 문장",
                placeholder="여기에 읽게 할 문장을 입력하세요",
                lines=3,
            )
            gen_btn = gr.Button("음성 생성", variant="primary")
            gen_out = gr.Audio(label="생성 결과", type="filepath")
            gen_status = gr.Textbox(label="상태", interactive=False)

        with gr.Tab("목소리 등록"):
            gr.Markdown(
                "마이크로 녹음하거나 파일을 올리세요. 3초 이상, 10~20초 정도가 무난합니다.\n"
                "대본을 비워 두면 음성에서 자동으로 받아씁니다."
            )
            reg_name = gr.Textbox(label="이름", placeholder="예: 엄마")
            reg_audio = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="참조 음성 (녹음 또는 업로드)",
            )
            reg_text = gr.Textbox(
                label="참조 대본 (비우면 자동 생성)",
                placeholder="녹음한 내용을 그대로 적거나, 비워 두세요",
                lines=3,
            )
            reg_overwrite = gr.Checkbox(label="같은 이름이 있으면 덮어쓰기", value=False)
            reg_btn = gr.Button("등록", variant="primary")
            reg_status = gr.Textbox(label="상태", interactive=False, lines=2)
            gr.Markdown("받아쓴 대본이 틀렸다면 위 칸에서 고친 뒤 아래 버튼을 누르세요.")
            reg_save_text = gr.Button("대본만 다시 저장")

    gen_refresh.click(ui_refresh, outputs=gen_name)
    gen_name.change(ui_info, inputs=gen_name, outputs=[gen_ref_audio, gen_ref_text])
    gen_btn.click(ui_generate, inputs=[gen_name, gen_text], outputs=[gen_out, gen_status])

    reg_btn.click(
        ui_register,
        inputs=[reg_name, reg_audio, reg_text, reg_overwrite],
        outputs=[reg_status, reg_text, gen_name],
    )
    reg_save_text.click(ui_save_text, inputs=[reg_name, reg_text], outputs=reg_status)

    demo.load(ui_info, inputs=gen_name, outputs=[gen_ref_audio, gen_ref_text])

# ── 로그인 정보 ─────────────────────────────────────────
# Colab 왼쪽 🔑 보안 비밀에 UI_USER, UI_PASS 를 등록해 두면 그 값을 쓴다.
# 없으면 세션마다 임시 비밀번호를 만들어 아래에 출력한다.
try:
    from google.colab import userdata
    AUTH = (userdata.get("UI_USER"), userdata.get("UI_PASS"))
except Exception:
    import secrets
    AUTH = ("user", secrets.token_urlsafe(8))
    print("보안 비밀 UI_USER / UI_PASS 가 없어 임시 로그인 정보를 만들었습니다.")
    print(f"  아이디: {AUTH[0]}    비밀번호: {AUTH[1]}")
    print()

demo.launch(
    share=True,                                    # 공개 주소를 만들되
    auth=AUTH,                                     # 로그인 없이는 들어올 수 없다
    inline=False,                                  # 셀 안 iframe은 쿠키 문제로 로그인이 안 되므로 끈다
    quiet=True,
    allowed_paths=[str(VOICE_DIR), str(OUTPUT_DIR)],
    show_error=True,
)

print("아래 주소를 새 탭에서 열고 로그인하세요.")
print(demo.share_url)